# Notebook 3: Model Training
**Autoencoders for Open-Set Presentation Attack Detection**

This notebook:
- Builds the DWT Denoising Autoencoder model
- Trains ONLY on bona fide (real) face images
- Uses mixed precision (FP16) for RTX 4050 optimization
- Implements early stopping with cosine annealing LR
- Saves best model checkpoint

## 1. Setup

In [1]:
import os
import sys
import torch
import numpy as np
import matplotlib.pyplot as plt

sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), '..')))
from src import config as cfg
from src.dataset import load_manifest, create_dataloaders
from src.model import build_model, DWTAutoencoder
from src.train import Trainer
from src.utils import set_seed, check_gpu, model_summary, plot_training_history

set_seed()
check_gpu()
cfg.print_config()

  Random seed set to 42

  GPU Information:
    CUDA available: True
    Device name: NVIDIA GeForce RTX 4090
    VRAM: 24.0 GB
    CUDA version: 12.4
    PyTorch version: 2.6.0+cu124
CONFIGURATION
  Device:            cuda
  Image Size:        128x128
  DWT Sub-band Size: 64x64
  Latent Dim:        512
  Batch Size:        32
  Epochs:            80
  Learning Rate:     0.0003
  Mixed Precision:   True
  Loss Weights:      α=0.3, β=1.5, γ=0.0
  Fusion Weight:     lambda=0.5
  Anomaly Weights:   w_iso=0.4, w_lmse=0.05, w_spec=0.4, w_maha=0.15
  IF Contamination:  0.05 (paper value)
  Train Users:       30 (IDs 1-30)
  Val Users:         7 (IDs 31-37)
  Test Users:        8 (IDs 38-45)
  Known Attacks:     [1]
  Unknown Attacks:   [2, 3, 4]


## 2. Load Data

In [2]:
# Load the manifest created in Notebook 02
manifest = load_manifest(cfg.MANIFEST_PATH)
print(f"Loaded manifest: {len(manifest)} entries")

# Create DataLoaders
train_loader, val_loader, test_loader = create_dataloaders(manifest)

Loaded manifest: 52650 entries
[Dataset] Split='train', bona_fide_only=False, samples=11700
[Dataset] Split='val', bona_fide_only=False, samples=8190
[Dataset] Split='test', bona_fide_only=False, samples=9360


## 3. Build Model

In [3]:
model = build_model()
model_summary(model)

# Verify with a dummy input
dummy_dwt = torch.randn(2, 12, 64, 64)
dummy_hh = torch.randn(2, 3, 64, 64)
with torch.no_grad():
    output = model(dummy_dwt, dummy_hh)

print(f"\nDummy forward pass:")
print(f"  z0 shape:      {output['z0'].shape}")
print(f"  z_recon shape:  {output['z_recon'].shape}")
print(f"  hh_recon shape: {output['hh_recon'].shape}")
print(f"  latent_mse:     {output['latent_mse'].shape}")
print(f"  spectral_l1:    {output['spectral_l1'].shape}")


  Model Summary:
    Total parameters:       17,028,387
    Trainable parameters:   17,028,387
    Model size (MB):             64.96 (FP32)
    Model size (MB):             32.48 (FP16)

  Components:
    encoder                   11,468,416 params
    denoiser                   2,760,192 params
    spectral_decoder           2,799,779 params

Dummy forward pass:
  z0 shape:      torch.Size([2, 512])
  z_recon shape:  torch.Size([2, 512])
  hh_recon shape: torch.Size([2, 3, 64, 64])
  latent_mse:     torch.Size([2])
  spectral_l1:    torch.Size([2])


## 4. VRAM Estimation

In [4]:
if torch.cuda.is_available():
    model_gpu = model.to(cfg.DEVICE)
    
    # Simulate a training step to measure VRAM
    torch.cuda.reset_peak_memory_stats()
    
    dummy_dwt = torch.randn(cfg.BATCH_SIZE, 12, 64, 64, device=cfg.DEVICE)
    dummy_hh = torch.randn(cfg.BATCH_SIZE, 3, 64, 64, device=cfg.DEVICE)
    
    from torch.cuda.amp import autocast, GradScaler
    scaler = GradScaler(enabled=cfg.USE_AMP)
    
    optimizer = torch.optim.AdamW(model_gpu.parameters(), lr=cfg.LEARNING_RATE)
    optimizer.zero_grad()
    
    with autocast(enabled=cfg.USE_AMP):
        output = model_gpu(dummy_dwt, dummy_hh)
        loss = output['latent_mse'].mean() + output['spectral_l1'].mean()
    
    scaler.scale(loss).backward()
    scaler.step(optimizer)
    scaler.update()
    
    peak_mem = torch.cuda.max_memory_allocated() / 1024**3
    total_mem = torch.cuda.get_device_properties(0).total_memory / 1024**3
    
    print(f"\nVRAM Usage (batch_size={cfg.BATCH_SIZE}, {'FP16' if cfg.USE_AMP else 'FP32'}):")
    print(f"  Peak VRAM used:  {peak_mem:.2f} GB")
    print(f"  Total VRAM:      {total_mem:.2f} GB")
    print(f"  Available:       {total_mem - peak_mem:.2f} GB")
    print(f"  Utilization:     {peak_mem/total_mem*100:.1f}%")
    
    if peak_mem < total_mem * 0.8:
        print(f"  âœ“ VRAM OK â€” plenty of headroom")
    else:
        print(f"  âš  VRAM tight â€” consider reducing batch size")
    
    # Clean up
    del dummy_dwt, dummy_hh, output, loss
    torch.cuda.empty_cache()
    model = model_gpu.cpu()

C:\Users\Shivang\AppData\Local\Temp\ipykernel_63164\42801444.py:11: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler(enabled=cfg.USE_AMP)
C:\Users\Shivang\AppData\Local\Temp\ipykernel_63164\42801444.py:16: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(enabled=cfg.USE_AMP):



VRAM Usage (batch_size=32, FP16):
  Peak VRAM used:  1.36 GB
  Total VRAM:      23.99 GB
  Available:       22.63 GB
  Utilization:     5.7%
  âœ“ VRAM OK â€” plenty of headroom


## 5. Train the Model

In [5]:
# Initialize trainer
trainer = Trainer(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    learning_rate=cfg.LEARNING_RATE,
    num_epochs=cfg.NUM_EPOCHS,
    device=cfg.DEVICE,
    checkpoint_dir=cfg.CHECKPOINT_DIR,
    use_amp=cfg.USE_AMP,
)

  [Encoder] Backbone frozen. Only FC head is trainable.


In [6]:
# Run training!
history = trainer.train()


TRAINING START  (Two-Phase Strategy)
  Total parameters:     17,028,387
  Trainable parameters: 5,823,651  (34.2%)
  Backbone frozen:      Phase-1 (epochs 1-20)
  Phase-2 unfreeze:     layer3+layer4 at epoch 21
  Base LR:              3.0e-04
  Phase-2 LR:           3.0e-05
  DELTA (compact):      0.1
  Device: cuda
  Mixed Precision: True
  Train batches: 365
  Val batches: 256



Epoch 1/80 [Train] [Phase-1]: 100%|██████████| 365/365 [00:43<00:00,  8.47it/s, loss=0.0195, mse=0.0239, l1=0.0082, cmpct=0.000]



  [Centroid Init] Computing mean bona-fide z0 from train set...


  [Centroid] Collecting z0: 100%|██████████| 365/365 [00:34<00:00, 10.73it/s]


  [CombinedPADLoss] Centroid initialized (norm=21.8834). Now frozen.


...


  Epoch 68/80 [Phase-2] (67.6 min elapsed)
    Train Loss: 0.015495 (MSE: 0.000186, L1: 0.010272, Compact: 0.0003)
    Val Loss:   0.003200 (ReconGap: +6.52e-3, SepRatio: 1.51std  [GOOD ✓])
    LR: 1.97e-04
    No improvement (18/20)



Epoch 69/80 [Val]: 100%|██████████| 256/256 [00:25<00:00, 10.14it/s, loss=0.000000]



  Epoch 69/80 [Phase-2] (68.6 min elapsed)
    Train Loss: 0.019062 (MSE: 0.000124, L1: 0.010320, Compact: 0.0056)
    Val Loss:   0.042980 (ReconGap: +6.83e-3, SepRatio: 1.58std  [GOOD ✓])
    LR: 1.74e-04
    No improvement (19/20)



Epoch 70/80 [Val]: 100%|██████████| 256/256 [00:24<00:00, 10.28it/s, loss=0.000000]



  Epoch 70/80 [Phase-2] (69.6 min elapsed)
    Train Loss: 0.017439 (MSE: 0.000098, L1: 0.010313, Compact: 0.0034)
    Val Loss:   0.003119 (ReconGap: +5.03e-3, SepRatio: 1.16std  [GOOD ✓])
    LR: 1.50e-04
    No improvement (20/20)

  Early stopping at epoch 70

TRAINING COMPLETE
  Total time: 69.6 minutes
  Best loss: 0.001841
  Training history saved to c:\BS_Shivang\checkpoints\training_history.json


## 6. Training Curves

In [9]:
fig = plot_training_history(
    history,
    save_path=os.path.join(cfg.RESULTS_DIR, 'training_curves.png')
)

# Display inline
fig2, axes = plt.subplots(1, 3, figsize=(18, 5))
epochs = range(1, len(history['train_loss']) + 1)

axes[0].plot(epochs, history['train_loss'], 'b-', label='Train', linewidth=2)
if any(v > 0 for v in history['val_loss']):
    axes[0].plot(epochs, history['val_loss'], 'r-', label='Val', linewidth=2)
axes[0].set_title('Total Loss')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(epochs, history['train_latent_mse'], 'b-', label='Train', linewidth=2)
axes[1].set_title('Latent MSE')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

axes[2].plot(epochs, history['train_spectral_l1'], 'b-', label='Train', linewidth=2)
axes[2].set_title('Spectral L1')
axes[2].legend()
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

  Training curves saved to c:\BS_Shivang\results\training_curves.png


C:\Users\Shivang\AppData\Local\Temp\ipykernel_63164\3761605861.py:28: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 7. Load Best Model (for verification)

In [10]:
# Verify best model loads correctly
best_model = build_model()
checkpoint = torch.load(
    os.path.join(cfg.CHECKPOINT_DIR, 'best_model.pth'),
    map_location='cpu'
)
best_model.load_state_dict(checkpoint['model_state_dict'])
print(f"âœ“ Best model loaded from epoch {checkpoint['epoch']+1}")
print(f"  Loss: {checkpoint['loss']:.6f}")

print("\nâœ“ Training complete. Proceed to Notebook 04 for evaluation.")

âœ“ Best model loaded from epoch 50
  Loss: 0.001841

âœ“ Training complete. Proceed to Notebook 04 for evaluation.
